# LSTM real-vs-fake classifier on HF-vs-HR (PolitiFact++ & GossipCop++)

Faithful port of `true-and-fake-news-lstm-accuracy-97-90.ipynb` (Keras LSTM, ~97.9% on the
Kaggle ISOT set), pointed at the LIFE **human-written** subset. **Task = fake-vs-real among
human news:** HF (human_fake) = **0 (fake)**, HR (human_true) = **1 (real)** — the same
Fake=0 / True=1 convention as the source notebook.

- Same pipeline: clean/lemmatize → Keras `Tokenizer` → `pad_sequences(150)` →
  `Embedding → LSTM(150) → GlobalMaxPool → Dense → softmax(2)`, Adam 1e-4, 15 epochs.
- **Methodological contrast:** HF-vs-HR is the task LIFE's fingerprint method used as a
  *negative control* (it found no signal — neither class is LLM-generated). A content-based LSTM
  can instead pick up topical/stylistic differences, so it may separate them where LIFE couldn't.
- **Caveats:** PolitiFact++ HF/HR is only **291 articles** → the LSTM will **overfit**; read the
  test numbers as noisy. GossipCop++ (**12,252**) is fine. Both are **1:2 fake:real**, so watch
  **fake(HF=0) recall** and the confusion matrix, not just accuracy. High accuracy may also
  reflect source/topic style rather than genuine deception (as on the ISOT benchmark itself).
- **GPU optional** (TF uses it automatically if present; speeds up GossipCop++).

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_DIR    = '/content/drive/MyDrive/LIFE'
DATASET_ROOT   = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR  = f'{DATASET_ROOT}/GossipCop++'

os.chdir(PROJECT_DIR)  # so the relative script path below resolves
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))
print('GossipCop++  found:', os.path.isdir(GOSSIPCOP_DIR))

## Run the classifier

Each cell prints the class balance, per-epoch train/val accuracy, then a test
`classification_report` + confusion matrix. PolitiFact++ is seconds; GossipCop++ (~9.8k train,
15 epochs) is a few minutes (faster on GPU).

In [ ]:
!python lstm_real_vs_fake_code/run_life_lstm.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++

In [ ]:
!python lstm_real_vs_fake_code/run_life_lstm.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++

## Notes
- **Task = fake-vs-real among human news** (HF=0 vs HR=1) — NOT AI-vs-Human and NOT LIFE's
  MF-vs-MR. To change the cut, edit `FILE_LABELS` in `run_life_lstm.py`.
- Faithful to the source notebook (maxlen 150, Embedding 100, LSTM 150, Adam 1e-4, 15 epochs);
  the only deviation is a **stratified** 80/20 split so PolitiFact++'s tiny test set keeps the
  1:2 ratio. No model checkpoints saved.
- Expect PolitiFact++ to hit near-perfect train accuracy with an unstable test score — that's the
  overfitting caveat, not a bug. GossipCop++ should be more stable.